<a href="https://colab.research.google.com/github/andressonsino/ds-toolkit/blob/main/machine-learning/carga_dataset.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 📦 Template: Carga de Dataset

Plantilla reutilizable para cargar datasets desde distintas fuentes.  
Funciona tanto en **JupyterLab local** como en **Google Colab**.

---

## ¿Cómo usar esta plantilla?

1. Usá la **Sección 1** si querés combinar múltiples fuentes con fallback automático
2. Usá la **Sección 2** si solo necesitás una fuente específica
3. Cambiá las variables marcadas con `# ← CAMBIAR`

---

# Sección 1 — Plantilla con Fallback Automático

Estructura `if/elif/else` para combinar fuentes en orden de prioridad.  
El sistema prueba cada opción y avanza a la siguiente si falla.

**Cuándo usarla:** cuando el proyecto debe ser reproducible en distintos entornos (local, Colab, equipo).

In [ ]:
import os
import pandas as pd

# ── Configuración ──────────────────────────────────────────
DATASET_KAGGLE_PROPIO   = "andrssonsinogrugni/nombre-dataset"  # ← CAMBIAR: tu dataset en Kaggle
DATASET_KAGGLE_EXTERNO  = "mlg-ulb/creditcardfraud"           # ← CAMBIAR: dataset de otro usuario
DATA_PATH_LOCAL         = r"data/archivo.csv"                 # ← CAMBIAR: ruta local
# ───────────────────────────────────────────────────────────

df = None

# ── Prioridad 1: archivo local ya descargado ───────────────
if os.path.exists(DATA_PATH_LOCAL):
    df = pd.read_csv(DATA_PATH_LOCAL)
    print(f"✅ Dataset cargado desde archivo local: {DATA_PATH_LOCAL}")

# ── Prioridad 2: tus propios datasets de Kaggle ────────────
if df is None:
    try:
        import subprocess
        subprocess.run(["pip", "install", "-q", "kagglehub"], check=True)
        import kagglehub
        path = kagglehub.dataset_download(DATASET_KAGGLE_PROPIO)
        archivos = os.listdir(path)
        print("Archivos encontrados:", archivos)
        df = pd.read_csv(f"{path}/{archivos[0]}")
        print("✅ Dataset cargado desde tus datasets de Kaggle")
    except Exception as e:
        print(f"⚠️ No se pudo cargar desde tu Kaggle: {e}")

# ── Prioridad 3: dataset externo de Kaggle ─────────────────
if df is None:
    try:
        import kagglehub
        path = kagglehub.dataset_download(DATASET_KAGGLE_EXTERNO)
        archivos = os.listdir(path)
        print("Archivos encontrados:", archivos)
        df = pd.read_csv(f"{path}/{archivos[0]}")
        print("✅ Dataset cargado desde dataset externo de Kaggle")
    except Exception as e:
        raise FileNotFoundError(
            f"No se pudo cargar el dataset desde ninguna fuente.\n"
            f"Opciones manuales:\n"
            f"  1. Colocá el archivo en: {DATA_PATH_LOCAL}\n"
            f"  2. Subí tu dataset a: https://www.kaggle.com/datasets/{DATASET_KAGGLE_PROPIO}\n"
            f"  3. Descargalo desde: https://www.kaggle.com/datasets/{DATASET_KAGGLE_EXTERNO}\n"
        )

print(f"\nDataset listo — Filas: {df.shape[0]} | Columnas: {df.shape[1]}")

---

# Sección 2 — Opciones Individuales

Usá una sola celda según la fuente que necesitás.

---

## Opción 1 — Kaggle (recomendada)

Descarga el dataset desde Kaggle y lo guarda en caché local.  
Si el dataset se actualiza en Kaggle, `kagglehub` lo re-descarga automáticamente.

**Cuándo usarla:** dataset de Kaggle, querés reproducibilidad sin descargar manualmente.

In [ ]:
import kagglehub
import pandas as pd
import os

# ── Configuración ──────────────────────────────────────────
DATASET_KAGGLE = "mlg-ulb/creditcardfraud"  # ← CAMBIAR: usuario/nombre-dataset
# ───────────────────────────────────────────────────────────

# Descarga a caché local (no re-descarga si ya está actualizado)
path = kagglehub.dataset_download(DATASET_KAGGLE)

# Detecta el archivo automáticamente
archivos = os.listdir(path)
print("Archivos descargados:", archivos)

# Carga el primer archivo CSV encontrado
df = pd.read_csv(f"{path}/{archivos[0]}")
print(f"✅ Dataset cargado — Filas: {df.shape[0]} | Columnas: {df.shape[1]}")

---

## Opción 2 — CSV local

**Cuándo usarla:** tenés el archivo descargado en tu máquina, no viene de Kaggle.

In [ ]:
import pandas as pd

# ── Configuración ──────────────────────────────────────────
DATA_PATH = r"data/archivo.csv"  # ← CAMBIAR: ruta al archivo
SEPARATOR = ","                  # ← CAMBIAR si es otro separador (";", "\t", etc.)
ENCODING  = "utf-8"              # ← CAMBIAR si hay problemas de caracteres
# ───────────────────────────────────────────────────────────

df = pd.read_csv(DATA_PATH, sep=SEPARATOR, encoding=ENCODING)
print(f"✅ Dataset cargado — Filas: {df.shape[0]} | Columnas: {df.shape[1]}")

---

## Opción 3 — Scikit-learn (datasets built-in)

**Cuándo usarla:** datasets clásicos como Iris, Wine, Breast Cancer, Digits.

In [ ]:
import pandas as pd
from sklearn.datasets import load_iris  # ← CAMBIAR: load_wine, load_breast_cancer, etc.

data = load_iris()
df = pd.DataFrame(data.data, columns=data.feature_names)
df['target'] = data.target

print(f"✅ Dataset cargado — Filas: {df.shape[0]} | Columnas: {df.shape[1]}")
print(f"Clases: {data.target_names}")

---

## Opción 4 — Keras (imágenes y series)

**Cuándo usarla:** MNIST, Fashion-MNIST, CIFAR-10, etc.

In [ ]:
import tensorflow as tf

# ── Configuración ──────────────────────────────────────────
dataset = tf.keras.datasets.mnist  # ← CAMBIAR: fashion_mnist, cifar10, cifar100, imdb, etc.
# ───────────────────────────────────────────────────────────

(X_train, y_train), (X_test, y_test) = dataset.load_data()

print(f"✅ Dataset cargado")
print(f"Train: {X_train.shape} | Test: {X_test.shape}")

---

## Opción 5 — URL directa

**Cuándo usarla:** dataset publicado en una URL pública (UCI, GitHub raw, etc.).

In [ ]:
import pandas as pd

# ── Configuración ──────────────────────────────────────────
URL = "https://raw.githubusercontent.com/usuario/repo/main/data.csv"  # ← CAMBIAR
# ───────────────────────────────────────────────────────────

df = pd.read_csv(URL)
print(f"✅ Dataset cargado — Filas: {df.shape[0]} | Columnas: {df.shape[1]}")